In [2]:
!pip install -q pyspark

In [1]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("SalesDataLake")
    .master("local[*]")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .getOrCreate()
)
spark.version

'4.0.5'

## 2. Create Data Lake structure and place raw file in Bronze

In [3]:
import os, shutil, hashlib

BASE = "data_lake"
BRONZE = f"{BASE}/bronze"
SILVER = f"{BASE}/silver/preprocessed_sales"
os.makedirs(BRONZE, exist_ok=True)
os.makedirs(SILVER, exist_ok=True)

if not os.path.exists("sales.csv"):
    from google.colab import files
    uploaded = files.upload()

shutil.copy("sales.csv", f"{BRONZE}/sales.csv")

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

bronze_hash_before = md5(f"{BRONZE}/sales.csv")
for root, dirs, fs in os.walk(BASE):
    print(root, fs)

Saving sales.csv to sales.csv
data_lake []
data_lake/bronze ['sales.csv']
data_lake/silver []
data_lake/silver/preprocessed_sales []


## 3. Load and inspect with Spark

In [4]:
bronze_path = f"{BRONZE}/sales.csv"
df_raw = spark.read.csv(bronze_path, header=True, inferSchema=True)
df_raw.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [5]:
df_raw.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [6]:
raw_count = df_raw.count()
print("Total records (Bronze):", raw_count)

Total records (Bronze): 1000


### 3.1 Duplicates

In [7]:
exact_dup_rows = raw_count - df_raw.dropDuplicates().count()
print("Exact duplicate rows:", exact_dup_rows)

dup_ids = df_raw.groupBy("order_id").count().filter("count > 1")
print("order_id values appearing more than once:", dup_ids.count())
df_raw.join(dup_ids.select("order_id"), "order_id").orderBy("order_id").show(12, truncate=False)

Exact duplicate rows: 10
order_id values appearing more than once: 18
+--------+-----------+--------------+----------------------+---------------+--------+----------+----------------+----------------+----------+-------------+----------+------------+
|order_id|customer_id|customer_name |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state        |order_date|order_status|
+--------+-----------+--------------+----------------------+---------------+--------+----------+----------------+----------------+----------+-------------+----------+------------+
|100055  |C0270      |Pranav Iyer   |Dry Fruits Pack       |Grocery        |7       |930       |12              |Cash on Delivery|Coimbatore|Tamil Nadu   |2026-08-20|Delivered   |
|100055  |C0270      |Pranav Iyer   |Dry Fruits Pack       |Grocery        |7       |930       |12              |Cash on Delivery|Coimbatore|Tamil Nadu   |2026-08-20|Delivered   |
|100064  |C0276      |Ananya A

### 3.2 Null / missing values

In [8]:
df_raw.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
    for c in df_raw.columns
]).show(truncate=False)

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



### 3.3 Text-format inconsistencies

In [9]:
text_cols = [c for c, t in df_raw.dtypes if t == "string"]

df_raw.select([
    F.sum(F.when(F.col(c).isNotNull() & (F.col(c) != F.trim(F.regexp_replace(F.col(c), r"\s+", " "))), 1).otherwise(0)).alias(c)
    for c in text_cols
]).show(truncate=False)

+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|customer_id|customer_name|product|category|payment_method|city|state|order_date|order_status|
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|0          |3            |0      |0       |0             |3   |0    |0         |0           |
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+



In [10]:
for c in ["category", "city", "state", "payment_method", "order_status"]:
    print(f"--- {c}: values not in standard Title Case format")
    (df_raw.filter(F.col(c).isNotNull() & (F.trim(F.col(c)) != F.initcap(F.trim(F.col(c)))))
        .groupBy(c).count().orderBy(F.desc("count")).show(truncate=False))

--- category: values not in standard Title Case format
+-----------+-----+
|category   |count|
+-----------+-----+
|beauty     |2    |
|SPORTS     |1    |
|ELECTRONICS|1    |
|sports     |1    |
|CLOTHING   |1    |
+-----------+-----+

--- city: values not in standard Title Case format
+----------+-----+
|city      |count|
+----------+-----+
|DELHI     |2    |
|delhi     |1    |
|HYDERABAD |1    |
|JAIPUR    |1    |
|coimbatore|1    |
+----------+-----+

--- state: values not in standard Title Case format
+-----+-----+
|state|count|
+-----+-----+
+-----+-----+

--- payment_method: values not in standard Title Case format
+----------------+-----+
|payment_method  |count|
+----------------+-----+
|Cash on Delivery|208  |
|UPI             |170  |
+----------------+-----+

--- order_status: values not in standard Title Case format
+------------+-----+
|order_status|count|
+------------+-----+
+------------+-----+



In [11]:
for c in ["category", "city"]:
    print(f"--- {c}: same value written with different spellings")
    (df_raw.filter(F.col(c).isNotNull())
        .groupBy(F.lower(F.trim(F.col(c))).alias("normalized"))
        .agg(F.countDistinct(c).alias("variants"), F.collect_set(c).alias("spellings"))
        .filter("variants > 1").show(truncate=False))

--- category: same value written with different spellings
+-----------+--------+--------------------------+
|normalized |variants|spellings                 |
+-----------+--------+--------------------------+
|beauty     |2       |[beauty, Beauty]          |
|electronics|2       |[ELECTRONICS, Electronics]|
|sports     |3       |[sports, Sports, SPORTS]  |
|clothing   |2       |[CLOTHING, Clothing]      |
+-----------+--------+--------------------------+

--- city: same value written with different spellings
+----------+--------+-----------------------------------+
|normalized|variants|spellings                          |
+----------+--------+-----------------------------------+
|delhi     |3       |[DELHI, delhi, Delhi]              |
|hyderabad |3       |[HYDERABAD, Hyderabad,  Hyderabad ]|
|mumbai    |2       |[ Mumbai , Mumbai]                 |
|jaipur    |2       |[Jaipur, JAIPUR]                   |
|coimbatore|2       |[Coimbatore, coimbatore]           |
+----------+--------+--

### 3.4 Invalid numerical values

In [12]:
print("quantity <= 0 :", df_raw.filter(F.col("quantity") <= 0).count())
print("unit_price <= 0:", df_raw.filter(F.col("unit_price") <= 0).count())
print("discount_percent < 0 or > 100:", df_raw.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).count())

df_raw.filter(
    (F.col("quantity") <= 0) | (F.col("unit_price") <= 0) |
    (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)
).select("order_id", "product", "quantity", "unit_price", "discount_percent").show(40, truncate=False)

quantity <= 0 : 12
unit_price <= 0: 8
discount_percent < 0 or > 100: 8
+--------+----------------------+--------+----------+----------------+
|order_id|product               |quantity|unit_price|discount_percent|
+--------+----------------------+--------+----------+----------------+
|100034  |Yoga Mat              |7       |1720      |-5              |
|100102  |Atta 10kg             |7       |0         |40              |
|100114  |Ceiling Fan           |0       |2920      |15              |
|100140  |Cooking Oil 5L        |4       |790       |-5              |
|100143  |Football              |0       |1030      |15              |
|100170  |Competitive Exam Guide|6       |0         |0               |
|100198  |Hoodie                |-2      |2160      |12              |
|100217  |Football              |1       |650       |-5              |
|100235  |Power Bank            |7       |-500      |0               |
|100284  |Study Table           |2       |-500      |30              |
|10031

In [13]:
df_raw.select("quantity", "unit_price", "discount_percent").summary("min", "25%", "50%", "75%", "max").show()

+-------+--------+----------+----------------+
|summary|quantity|unit_price|discount_percent|
+-------+--------+----------+----------------+
|    min|      -2|      -500|              -5|
|    25%|       3|       730|              10|
|    50%|       5|      1790|              20|
|    75%|       8|      4870|              30|
|    max|      10|     81010|             120|
+-------+--------+----------+----------------+



### 3.5 Inconsistent / invalid dates

In [14]:
date_formats = {
    "yyyy-MM-dd": r"^\d{4}-\d{2}-\d{2}$",
    "dd/MM/yyyy": r"^\d{2}/\d{2}/\d{4}$",
    "yyyy/MM/dd": r"^\d{4}/\d{2}/\d{2}$",
    "MM-dd-yyyy": r"^\d{2}-\d{2}-\d{4}$",
}

date_str = F.trim(F.col("order_date").cast("string"))
fmt_col = F.lit("unrecognized")
for fmt, rx in reversed(list(date_formats.items())):
    fmt_col = F.when(date_str.rlike(rx), F.lit(fmt)).otherwise(fmt_col)

df_raw.withColumn("detected_format", fmt_col).groupBy("detected_format").count().show()

+---------------+-----+
|detected_format|count|
+---------------+-----+
|   unrecognized|    1|
|     yyyy/MM/dd|    1|
|     yyyy-MM-dd|  996|
|     MM-dd-yyyy|    1|
|     dd/MM/yyyy|    1|
+---------------+-----+



In [15]:
parsed = F.coalesce(*[
    F.when(date_str.rlike(rx), F.to_date(date_str, fmt)) for fmt, rx in date_formats.items()
])

df_raw.withColumn("detected_format", fmt_col).withColumn("parsed_date", parsed) \
    .filter((F.col("detected_format") != "yyyy-MM-dd") | F.col("parsed_date").isNull()) \
    .select("order_id", "order_date", "detected_format", "parsed_date").show(truncate=False)

+--------+----------+---------------+-----------+
|order_id|order_date|detected_format|parsed_date|
+--------+----------+---------------+-----------+
|100097  |31/02/2026|dd/MM/yyyy     |NULL       |
|100358  |2025/07/14|yyyy/MM/dd     |2025-07-14 |
|100448  |not-a-date|unrecognized   |NULL       |
|100634  |2025-02-30|yyyy-MM-dd     |NULL       |
|100637  |2026-13-05|yyyy-MM-dd     |NULL       |
|100969  |09-15-2026|MM-dd-yyyy     |2026-09-15 |
+--------+----------+---------------+-----------+



## 4. Preprocessing - Silver layer

### 4.1 Remove exact duplicate rows

In [16]:
df = df_raw.dropDuplicates()
print("Rows after removing exact duplicates:", df.count())

Rows after removing exact duplicates: 990


### 4.2 Trim spaces and standardize capitalization

In [17]:
for c in text_cols:
    df = df.withColumn(c, F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
    df = df.withColumn(c, F.when(F.col(c) == "", F.lit(None)).otherwise(F.col(c)))

for c in ["customer_name", "category", "city", "state"]:
    df = df.withColumn(c, F.initcap(F.col(c)))

df.select("category", "city").distinct().orderBy("category", "city").show(60)

+-----------+----------+
|   category|      city|
+-----------+----------+
|     Beauty| Bengaluru|
|     Beauty|   Chennai|
|     Beauty|Coimbatore|
|     Beauty|     Delhi|
|     Beauty| Hyderabad|
|     Beauty|    Jaipur|
|     Beauty|   Kolkata|
|     Beauty|   Lucknow|
|     Beauty|    Mumbai|
|     Beauty|      Pune|
|     Beauty|Vijayawada|
|      Books|      NULL|
|      Books| Bengaluru|
|      Books|   Chennai|
|      Books|Coimbatore|
|      Books|     Delhi|
|      Books| Hyderabad|
|      Books|    Jaipur|
|      Books|   Kolkata|
|      Books|   Lucknow|
|      Books|    Mumbai|
|      Books|      Pune|
|      Books|Vijayawada|
|   Clothing|      NULL|
|   Clothing| Bengaluru|
|   Clothing|   Chennai|
|   Clothing|Coimbatore|
|   Clothing|     Delhi|
|   Clothing| Hyderabad|
|   Clothing|    Jaipur|
|   Clothing|   Kolkata|
|   Clothing|   Lucknow|
|   Clothing|    Mumbai|
|   Clothing|      Pune|
|   Clothing|Vijayawada|
|Electronics| Bengaluru|
|Electronics|   Chennai|


### 4.3 Validate numerical columns

In [18]:
df = (df.withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double")))

df = df.withColumn("quantity", F.when(F.col("quantity") > 0, F.col("quantity")))
df = df.withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")))
df = df.withColumn(
    "discount_percent",
    F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent")).otherwise(F.lit(0.0))
)

median_qty = df.approxQuantile("quantity", [0.5], 0.001)[0]
median_price = df.approxQuantile("unit_price", [0.5], 0.001)[0]

product_median = (
    df.groupBy("product")
      .agg(F.percentile_approx("unit_price", 0.5).alias("product_median_price"))
)

df = (df.join(product_median, "product", "left")
        .withColumn("unit_price",
                    F.coalesce("unit_price", "product_median_price", F.lit(median_price)))
        .drop("product_median_price")
        .withColumn("quantity", F.coalesce("quantity", F.lit(int(median_qty)).cast("int"))))

print("Median quantity used:", median_qty, "| Median unit price (fallback):", median_price)

Median quantity used: 5.0 | Median unit price (fallback): 1840.0


### 4.4 Handle missing values

In [19]:
df = df.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown",
})

df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)

+-------+--------+-----------+-------------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|product|order_id|customer_id|customer_name|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+-------+--------+-----------+-------------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0      |0       |0          |0            |0       |0       |0         |0               |0             |0   |0    |0         |0           |
+-------+--------+-----------+-------------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



### 4.5 Standardize dates to yyyy-MM-dd

In [20]:
date_str = F.col("order_date").cast("string")
parsed = F.coalesce(*[
    F.when(date_str.rlike(rx), F.to_date(date_str, fmt)) for fmt, rx in date_formats.items()
])

df = df.withColumn("order_date", F.date_format(parsed, "yyyy-MM-dd"))

print("Rows with invalid date (kept with null order_date):", df.filter(F.col("order_date").isNull()).count())
df.filter(F.col("order_date").isNull()).show(truncate=False)

Rows with invalid date (kept with null order_date): 4
+-------------+--------+-----------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|product      |order_id|customer_id|customer_name|category |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+-------------+--------+-----------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|Sofa         |100097  |C0244      |Shreya Rao   |Furniture|4       |47430.0   |10.0            |Net Banking     |Hyderabad|Telangana    |NULL      |Returned    |
|Novel        |100637  |C0020      |Aarav Patel  |Books    |4       |610.0     |20.0            |Net Banking     |Chennai  |Tamil Nadu   |NULL      |Delivered   |
|Bedside Table|100448  |C0270      |Pranav Iyer  |Furniture|1       |4290.0    |35.0            |UPI             |Lucknow  |Uttar P

### 4.6 Final column order and save to Silver

In [21]:
df_silver = df.select(df_raw.columns).orderBy("order_id")
df_silver.cache()
silver_count = df_silver.count()
print("Rows ready for Silver:", silver_count)

Rows ready for Silver: 990


In [22]:
(df_silver.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(SILVER))

os.listdir(SILVER)

['_SUCCESS',
 '.part-00000-2d37b830-f508-4d02-a9c2-dfb05a836bc4-c000.csv.crc',
 'part-00000-2d37b830-f508-4d02-a9c2-dfb05a836bc4-c000.csv',
 '._SUCCESS.crc']

In [23]:
part_file = [f for f in os.listdir(SILVER) if f.startswith("part-") and f.endswith(".csv")][0]
silver_csv = f"{BASE}/silver/sales_silver.csv"
shutil.copy(f"{SILVER}/{part_file}", silver_csv)

for root, dirs, fs in os.walk(BASE):
    print(root, fs)

data_lake []
data_lake/bronze ['sales.csv']
data_lake/silver ['sales_silver.csv']
data_lake/silver/preprocessed_sales ['_SUCCESS', '.part-00000-2d37b830-f508-4d02-a9c2-dfb05a836bc4-c000.csv.crc', 'part-00000-2d37b830-f508-4d02-a9c2-dfb05a836bc4-c000.csv', '._SUCCESS.crc']


## 5. Verification

In [24]:
print("Row count BEFORE preprocessing (Bronze):", raw_count)
print("Row count AFTER preprocessing (Silver) :", silver_count)
print("Rows removed                           :", raw_count - silver_count)

Row count BEFORE preprocessing (Bronze): 1000
Row count AFTER preprocessing (Silver) : 990
Rows removed                           : 10


In [25]:
df_check = spark.read.csv(SILVER, header=True, inferSchema=True)
df_check.printSchema()
df_check.show(10, truncate=False)
print("Rows read back from Silver:", df_check.count())

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+------------

In [26]:
print("Bronze file unchanged:", md5(f"{BRONZE}/sales.csv") == bronze_hash_before)
print("Bronze identical to original upload:", md5(f"{BRONZE}/sales.csv") == md5("sales.csv"))

Bronze file unchanged: True
Bronze identical to original upload: True


In [27]:
print("quantity <= 0            :", df_check.filter("quantity <= 0").count())
print("unit_price <= 0          :", df_check.filter("unit_price <= 0").count())
print("discount outside 0-100   :", df_check.filter("discount_percent < 0 OR discount_percent > 100").count())
print("null order_date          :", df_check.filter("order_date IS NULL").count())
print("distinct category values :", df_check.select("category").distinct().count())
print("distinct city values     :", df_check.select("city").distinct().count())

quantity <= 0            : 0
unit_price <= 0          : 0
discount outside 0-100   : 0
null order_date          : 4
distinct category values : 8
distinct city values     : 12


## 6. Preprocessing summary
1. Bronze layer holds the original `sales.csv` untouched.
2. Exact duplicate rows were removed with `dropDuplicates()`.
3. Extra spaces were trimmed and repeated spaces collapsed in all text columns; empty strings were converted to nulls.
4. Capitalization of `customer_name`, `category`, `city` and `state` was standardized with `initcap`.
5. `quantity <= 0` and `unit_price <= 0` were treated as invalid; quantity was replaced with the median quantity and unit price with the median price of the same product.
6. `discount_percent` outside 0-100 was reset to 0.
7. Missing `customer_name`, `payment_method` and `city` were filled with `Unknown`.
8. `order_date` values in multiple formats were converted to `yyyy-MM-dd`; impossible dates were set to null and the rows were kept.
9. The result was saved to `data_lake/silver/preprocessed_sales/` and copied as `data_lake/silver/sales_silver.csv`.